# The LLM Gateway - Portkey

---

### The problem, in one picture

Without a gateway, every app talks to every provider directly:

```
                    ┌──────────► OpenAI      (openai SDK,   key #1)
   Your app ────────┼──────────► Anthropic   (anthropic SDK, key #2)
                    └──────────► Gemini      (google SDK,   key #3)
```

So you end up writing:

* 3 different SDKs, 3 different response shapes
* 3 API keys pasted in 3 places
* No single place to see cost, latency, or errors
* No retry. If OpenAI hiccups, your app is down

With a gateway, there is **one door**:

```
                         ┌─────────────┐  ┌──────► OpenAI
   Your app ────────────►│   PORTKEY   │──┼──────► Anthropic
      (one SDK)          │   GATEWAY   │  └──────► Gemini
                         └──────┬──────┘
                                │ and for free you get:
                                ├─   logs, cost, latency
                                ├─   retry + fallback
                                ├─   load balancing
                                └─   caching
```

> ## **A gateway is a smart middleman between your app and the model providers.**

---

So you end up writing:

* 3 different SDKs, 3 different response shapes
* 3 API keys pasted in 3 places
* No single place to see cost, latency, or errors
* No retry. If OpenAI hiccups, your app is down

---

With a gateway, there is **one door**:

```
                         ┌─────────────┐  ┌──────► OpenAI
   Your app ────────────►│   PORTKEY   │──┼──────► Anthropic
      (one SDK)          │   GATEWAY   │  └──────► Gemini
                         └──────┬──────┘
                                │ and for free you get:
                                ├─   logs, cost, latency
                                ├─   retry + fallback
                                ├─   load balancing
                                └─   caching
```

> ## **A gateway is a smart middleman between your app and the model providers.**

---

# What is Portkey, and why use it?

### The 5 jobs a gateway does for you

| Job                | Without a gateway                   | With Portkey                              |
| ------------------ | ----------------------------------- | ----------------------------------------- |
| **One API**        | A different SDK per provider        | One SDK, swap models by changing a string |
| **Observability**  | You have no idea what it costs      | Every call logged: cost, tokens, latency  |
| **Reliability**    | One provider is down = you are down | Retry, then fall back to another model    |
| **Load balancing** | All traffic to one place            | Split traffic by weight                   |
| **Caching**        | Pay twice for the same question     | Repeat answers are free and instant       |

---

User Tracing

### The problem

10,000 requests are in your logs. A customer emails: *"your bot gave me a wrong answer
this morning."* Which of the 10,000 was theirs?

### The fix: label every request

Portkey gives you two labels:

| Label          | What it is                          | Use it for                                   |
| -------------- | ----------------------------------- | -------------------------------------------- |
| **`trace_id`** | Your own ID for **one user action** | Grouping the 3 calls that made up one answer |
| **`metadata`** | Any key/value labels you like       | Filtering by user, feature, environment      |

**`_user` is a special metadata key.** Portkey uses it to build per-user analytics
(cost per user, requests per user), so always set it.

---

# Retry, Timeout and Fallback

### Everything from here is one idea: the **config**

A **config** is a small dictionary that tells the gateway how to behave.
You hand it to the client, and the gateway does the work. **Your calling code never changes.**

```python
client = Portkey(api_key=..., config={ ... the rules ... })
```

That is the whole mental model. Now let's write four of them.

### 4.1 - Retry

LLM APIs return **429 (rate limited)** and **500 (server error)** all the time.
Without a gateway you write try/except loops. With one, you write this:

Timeout ⏱

A model that hangs for 60 seconds is worse than one that fails fast.
Set a deadline and move on.

> **The unit is milliseconds**, not seconds. `request_timeout: 10000` is 10 seconds.

> **Why this matters:** a timeout is a **408**, and a 408 can trigger a fallback.
> So "wait 10 seconds, then try a different model" is just two config keys together.

### 4.3 - Fallback

**If the first model fails, automatically try the second one.** No code change.

Here the first target is a model name that does not exist, so it will fail and the
gateway will quietly use the second one.

---

# Part 5 - Load balancing and Caching

### 5.1 - Load balancing

**Split traffic between models by weight.**

Why you would want this:

* Send 80% of traffic to the cheap model, 20% to the smart one
* A/B test two models on real users
* Spread load across two accounts to avoid rate limits

Weights are **relative**, so they do not need to add up to 1.

### 5.2 - Caching

**If the same question comes in twice, do not pay for it twice.**

| Mode         | Matches when                         | Use for                            |
| ------------ | ------------------------------------ | ---------------------------------- |
| `"simple"`   | The request is **exactly** the same  | FAQs, fixed prompts, testing       |
| `"semantic"` | The request **means** the same thing | Real user chat, worded differently |

> **`max_age` is in SECONDS** (unlike `request_timeout`, which is milliseconds).
> Minimum 60, maximum 7776000 (90 days), default 604800 (7 days).


In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

True

In [18]:
portkey_model = os.getenv("PORTKEY_MODEL")
portkey_api_key = os.getenv("PORTKEY_API_KEY")
portkey_provider = os.getenv("PORTKEY_PROVIDER")

In [ ]:
from portkey_ai import Portkey

client = Portkey(
    api_key=portkey_api_key,
    provider=portkey_provider
)

response = client.chat.completions.create(
    model=os.getenv("LLM_MODEL"),
    api_key=os.getenv("LLM_API_KEY"),
    messages=[{"role": "user", "content": "Say hello in exactly 5 words."}]
)

In [20]:
print(response.choices[0].message.content)

Greetings, wonderful world, hello again.


In [21]:
print("answer: ", response.choices[0].message.content)
print("model: ", response.model)
print("tokens: ", response.usage.total_tokens)

answer:  Greetings, wonderful world, hello again.
model:  openai/gpt-oss-20b
tokens:  260


In [22]:
def ask(client, question):
    """Send one question through the gateway and return the response object."""
    return client.chat.completions.create(
        model=portkey_model,
        messages=[{"role": "user", "content": question}]
    )

response = ask(client, "What is the API gateway? One short answer")
print(response.choices[0].message.content)

An API gateway is a server that acts as the single entry point for all client requests, routing calls to the appropriate microservices, handling cross‑cutting concerns (auth, rate limiting, caching, logging), and often transforming or aggregating responses before sending them back to the client.


In [23]:
response = client.with_options(
    trace_id="chat_001",
    metadata={"_user": "balaji.p@xyz.com", "feature": "support-bot"},
).chat.completions.create(
    model=portkey_model,
    messages=[{"role": "user", "content": "How do I reset my password?"}]
)
print(response.choices[0].message.content)

Sure! The steps are slightly different depending on the service you’re using, but most password‑reset flows follow the same pattern. Pick the one that matches the platform you’re on:

---

## 1. Web / Desktop Account

| Step | What to do | Tips |
|------|------------|------|
| **1. Go to the login page** | Open the website or app and click the “Log In” button. | Make sure you’re on the official site (check the URL). |
| **2. Click “Forgot password?”** | Usually located under or next to the password field. | Some sites call it “Reset password,” “Can’t access your account?” or “Trouble logging in?” |
| **3. Enter your email/username** | The address or handle you used to create the account. | If you can’t remember it, try the email you usually check. |
| **4. Check your inbox** | Look for an email titled something like “Password reset instructions” or “Reset your password.” | If it’s not in your inbox, check **Spam / Junk**. |
| **5. Click the reset link** | Opens a secure page to create 

# Retry

In [24]:
retry_config = {
    "retry": {
        "attempts": 3,
        "on_status_code": [429, 500, 502, 503]
    }
}

retry_client = Portkey(
    api_key=portkey_api_key,
    provider=portkey_provider,
    config="pc-gatewa-b3ee35",
)


response = ask(retry_client, "Say OK.")
print(response.choices[0].message.content)
print("Retries used: ", response.get_headers().get("retry-attempt-count"))

OK
Retries used:  0


# TimeOut

In [25]:
timeout_config = {
    "request_timeout": 10000,
    
}

timeout_client = Portkey(
    api_key=portkey_api_key,
    provider=portkey_provider,
    config="pc-gatewa-516860",
)


response = ask(timeout_client, "Say OK.")
print(response.choices[0].message.content)

OK.


In [26]:
impatient_config = {
    "request_timeout": 1
}

timeout_client = Portkey(
    api_key=portkey_api_key,
    provider=portkey_provider,
    config="pc-gatewa-f2be47",
)

try:
    response = ask(timeout_client, "Say OK.")
    print(response.choices[0].message.content)
    
except Exception as error:
    print("Time out, as expected")
    print("error:", str(error)[:120])

Time out, as expected
error: Error code: 408 - {'error': {'message': 'Request exceeded the timeout sent in the request: 1ms', 'type': 'timeout_error'


# FallBack

In [28]:
fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": os.getenv("PORTKEY_PROVIDER") + "/this-model-does-not-exist"}},
        {"override_params": {"model": os.getenv("PORTKEY_PROVIDER") + "/" + os.getenv("PORTKEY_MODEL")}},
    ],
}

fall_client = Portkey(
    api_key=portkey_api_key,
    config="pc-gatewa-c3f182"
)

response = fall_client.chat.completions.create(
    messages=[{"role": "user", "content": "Say OK."}]
)


print("answer:", response.choices[0].message.content)
print("Model used:", response.model)
print("Target index: ", response.get_headers().get("last-used-option-index"))

answer: OK.
Model used: openai/gpt-oss-20b
Target index:  config.targets[0]


# Load balancing and caching

In [31]:
load_balancing = {
    "strategy": {"model": "loadbalance"},
    "targets": [
        {"override_paramas": {"model": "@test/openai/gpt-oss-120b"}, "weight": 0.8},
        {"override_paramas": {"model": "@test/openai/gpt-oss-20b"}, "weight": 0.2},    
    ]
}

balanced_client = Portkey(
    api_key=portkey_api_key,
    config="pc-gatewa-d0abf0"
)

for i in range(10):
    response = balanced_client.chat.completions.create(
        messages=[{"role": "user", "content": "Say OK."}]
    )
    print(i + 1, "->", response.model)

1 -> openai/gpt-oss-120b
2 -> openai/gpt-oss-120b
3 -> openai/gpt-oss-120b
4 -> openai/gpt-oss-120b
5 -> openai/gpt-oss-120b
6 -> openai/gpt-oss-20b
7 -> openai/gpt-oss-120b
8 -> openai/gpt-oss-20b
9 -> openai/gpt-oss-120b
10 -> openai/gpt-oss-120b


# Caching

In [32]:
cache_config = {
    "cache": {
        "mode": "simple",
        "max_age": 300,
    },
}

cache_client = Portkey(
    api_key=portkey_api_key,
    provider=portkey_provider,
    config="pc-gatewa-a8a86a"
)

In [35]:
import time

question = "Explain how rainbow appears?"

for attempts in [1, 2]:
    start = time.time()
    
    response = ask(cache_client, question)
    end = time.time() - start
    
    seconds = round(end, 2)
    
    status = response.get_headers().get("cache-status")
    
    print("call", attempts, "took", seconds, "seconds| cache:", status)

call 1 took 0.06 seconds| cache: HIT
call 2 took 0.03 seconds| cache: HIT
